# Analyse statistique complète - Segmentation comportementale

**Pipeline complet :**
1. Chargement des données
2. Analyse en Composantes Principales (ACP)
3. Clustering non-supervisé (K-Means)
4. Analyse Discriminante Linéaire (LDA)
5. Optimisation du nombre de questions
6. Sous-segmentation

In [ ]:
%%sql -r df
-- Chargement des données consolidées
SELECT * FROM SNOWFLAKE_LEARNING_DB.PUBLIC.V_PERSONAS_COMPLET

In [ ]:
# Vérification des données
print(f"Nombre de personas: {len(df)}")
print(f"\nColonnes disponibles:")
for col in df.columns:
    print(f"  - {col}")

print(f"\nDistribution type décideur:")
print(df['TYPE_DECIDEUR'].value_counts())

In [ ]:
# ============================================================================
# 2. ANALYSE EN COMPOSANTES PRINCIPALES (ACP)
# ============================================================================
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import numpy as np

# Extraction des 4 axes comportementaux
X = df[['AXE_A', 'AXE_B', 'AXE_C', 'AXE_D']].values

# ACP
pca = PCA(n_components=4)
X_pca = pca.fit_transform(X)

# Résultats
print("ANALYSE EN COMPOSANTES PRINCIPALES")
print("="*50)
print(f"\nVariance expliquée par composante:")
for i, var in enumerate(pca.explained_variance_ratio_):
    print(f"  PC{i+1}: {var*100:.1f}%")
print(f"\nTotal (2 premières): {sum(pca.explained_variance_ratio_[:2])*100:.1f}%")

# Loadings (contribution des axes originaux)
print(f"\nLoadings (contribution des axes):")
print(f"{'':12} {'PC1':>8} {'PC2':>8}")
print("-"*30)
for i, ax in enumerate(['Axe A', 'Axe B', 'Axe C', 'Axe D']):
    print(f"{ax:12} {pca.components_[0, i]:>8.3f} {pca.components_[1, i]:>8.3f}")

In [ ]:
# Visualisation ACP
df['PC1'] = X_pca[:, 0]
df['PC2'] = X_pca[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Scatter plot coloré par archétype
ax1 = axes[0]
archetype_colors = {'Le Pilote à Vue': '#3498db', 'Le Pompier Épuisé': '#e74c3c', 
                    'Le Contrôleur Frustré': '#2ecc71', 'Le Converti Récent': '#f39c12',
                    'Le Gardien du Temple': '#9b59b6', "L'Humaniste des Équipes": '#1abc9c',
                    'Le Stratège en Construction': '#e91e63'}

for arch in df['ARCHETYPE'].unique():
    mask = df['ARCHETYPE'] == arch
    ax1.scatter(df.loc[mask, 'PC1'], df.loc[mask, 'PC2'],
                c=archetype_colors.get(arch, '#999'), 
                label=f"{arch.replace('Le ', '').replace("L'", '')} (n={mask.sum()})",
                alpha=0.6, s=60)

ax1.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)', fontsize=12)
ax1.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)', fontsize=12)
ax1.set_title('ACP - Personas par archétype', fontsize=13, fontweight='bold')
ax1.legend(loc='best', fontsize=8)
ax1.grid(True, alpha=0.3)
ax1.axhline(y=0, color='gray', linestyle='--', linewidth=0.5)
ax1.axvline(x=0, color='gray', linestyle='--', linewidth=0.5)

# Cercle des corrélations
ax2 = axes[1]
loadings = pca.components_.T
for i, ax_name in enumerate(['Axe A', 'Axe B', 'Axe C', 'Axe D']):
    ax2.arrow(0, 0, loadings[i, 0]*3, loadings[i, 1]*3, 
              head_width=0.15, head_length=0.1, fc='red', ec='red')
    ax2.text(loadings[i, 0]*3.3, loadings[i, 1]*3.3, ax_name, 
             fontsize=12, ha='center', fontweight='bold')

ax2.set_xlim(-4, 4)
ax2.set_ylim(-4, 4)
ax2.axhline(y=0, color='gray', linestyle='--', linewidth=0.5)
ax2.axvline(x=0, color='gray', linestyle='--', linewidth=0.5)
ax2.set_xlabel('PC1', fontsize=12)
ax2.set_ylabel('PC2', fontsize=12)
ax2.set_title('Cercle des corrélations', fontsize=13, fontweight='bold')
ax2.set_aspect('equal')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# 3. CLUSTERING NON-SUPERVISÉ (K-MEANS)
# ============================================================================
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Recherche du nombre optimal de clusters
print("RECHERCHE DU NOMBRE OPTIMAL DE CLUSTERS")
print("="*50)

inertias = []
silhouettes = []
K_range = range(2, 10)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X, km.labels_))

print(f"\nScores silhouette:")
for k, sil in zip(K_range, silhouettes):
    bar = '█' * int(sil * 30)
    print(f"  k={k}: {sil:.3f} {bar}")

best_k = K_range[np.argmax(silhouettes)]
print(f"\n→ Meilleur k selon silhouette: {best_k}")

In [ ]:
# Clustering avec k=4 (compromis interpétabilité/qualité)
k_optimal = 4
km = KMeans(n_clusters=k_optimal, random_state=42, n_init=10)
df['CLUSTER'] = km.fit_predict(X)

print(f"CLUSTERING K-MEANS (k={k_optimal})")
print("="*50)

# Profil de chaque cluster
print(f"\nProfil moyen par cluster:")
print(f"{'Cluster':>10} {'n':>6} {'A':>6} {'B':>6} {'C':>6} {'D':>6}")
print("-"*46)
for c in range(k_optimal):
    mask = df['CLUSTER'] == c
    n = mask.sum()
    a = df.loc[mask, 'AXE_A'].mean()
    b = df.loc[mask, 'AXE_B'].mean()
    cc = df.loc[mask, 'AXE_C'].mean()
    d = df.loc[mask, 'AXE_D'].mean()
    print(f"{'Cluster '+str(c+1):>10} {n:>6} {a:>6.1f} {b:>6.1f} {cc:>6.1f} {d:>6.1f}")

# Correspondance avec archétypes
print(f"\nArchétypes dominants par cluster:")
for c in range(k_optimal):
    mask = df['CLUSTER'] == c
    top_arch = df.loc[mask, 'ARCHETYPE'].value_counts().head(3)
    print(f"\n  Cluster {c+1}:")
    for arch, count in top_arch.items():
        print(f"    - {arch}: {count}")

In [ ]:
# Visualisation clusters
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

cluster_colors = {0: '#e74c3c', 1: '#2ecc71', 2: '#3498db', 3: '#f39c12'}
cluster_names = {0: 'Analytique', 1: 'Équilibré', 2: 'Réactif', 3: 'Collectif'}

ax1 = axes[0]
for c in range(k_optimal):
    mask = df['CLUSTER'] == c
    ax1.scatter(df.loc[mask, 'PC1'], df.loc[mask, 'PC2'],
                c=cluster_colors[c], label=f"Cluster {c+1} (n={mask.sum()})",
                alpha=0.6, s=60)

centroids_pca = pca.transform(km.cluster_centers_)
ax1.scatter(centroids_pca[:, 0], centroids_pca[:, 1], 
            c='black', marker='X', s=200, label='Centroïdes')

ax1.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)', fontsize=12)
ax1.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)', fontsize=12)
ax1.set_title('K-Means Clustering', fontsize=13, fontweight='bold')
ax1.legend(loc='best', fontsize=9)
ax1.grid(True, alpha=0.3)

# Heatmap correspondance
import seaborn as sns
ax2 = axes[1]
crosstab = df.groupby(['CLUSTER', 'ARCHETYPE']).size().unstack(fill_value=0)
crosstab.index = [f'Cluster {i+1}' for i in crosstab.index]
sns.heatmap(crosstab, annot=True, fmt='d', cmap='YlOrRd', ax=ax2)
ax2.set_title('Correspondance Clusters ↔ Archétypes', fontsize=13, fontweight='bold')
ax2.set_xlabel('')
ax2.set_ylabel('')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# 4. ANALYSE DISCRIMINANTE LINÉAIRE (LDA)
# ============================================================================
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import cross_val_score

# Préparation
y = df['TYPE_DECIDEUR'].values
mask_valid = ~df['TYPE_DECIDEUR'].isna()
X_valid = X[mask_valid]
y_valid = y[mask_valid]

le = LabelEncoder()
y_encoded = le.fit_transform(y_valid)

# LDA
lda = LinearDiscriminantAnalysis(n_components=2)
X_lda = lda.fit_transform(X_valid, y_encoded)

print("ANALYSE DISCRIMINANTE LINÉAIRE (LDA)")
print("="*50)
print(f"\nVariable cible: Type de décideur (Q10)")
print(f"Classes: {list(le.classes_)}")

print(f"\nVariance expliquée:")
for i, var in enumerate(lda.explained_variance_ratio_):
    print(f"  LD{i+1}: {var*100:.1f}%")

# Score
accuracy = lda.score(X_valid, y_encoded)
scores_cv = cross_val_score(lda, X_valid, y_encoded, cv=5)

print(f"\nPerformance:")
print(f"  Accuracy: {accuracy*100:.1f}%")
print(f"  Cross-validation (5-fold): {scores_cv.mean()*100:.1f}% (±{scores_cv.std()*100:.1f}%)")

# Profil moyen par classe
print(f"\nProfil moyen par type de décideur:")
print(f"{'Type':>20} {'n':>5} {'A':>6} {'B':>6} {'C':>6} {'D':>6}")
print("-"*50)
for i, label in enumerate(le.classes_):
    print(f"{label:>20} {(y_valid==label).sum():>5} {lda.means_[i, 0]:>6.1f} {lda.means_[i, 1]:>6.1f} {lda.means_[i, 2]:>6.1f} {lda.means_[i, 3]:>6.1f}")

In [ ]:
# Visualisation LDA
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

decideur_colors = {'Décideur autonome': '#2ecc71', 'Décideur contraint': '#e74c3c',
                   'Décideur partiel': '#f39c12', 'Non-décideur': '#9b59b6'}

# Scatter plot
ax1 = axes[0]
for label in le.classes_:
    mask_label = y_valid == label
    ax1.scatter(X_lda[mask_label, 0], X_lda[mask_label, 1],
                c=decideur_colors.get(label, '#999'), 
                label=f"{label} (n={mask_label.sum()})",
                alpha=0.7, s=70)

ax1.set_xlabel(f'LD1 ({lda.explained_variance_ratio_[0]*100:.1f}%)', fontsize=12)
ax1.set_ylabel(f'LD2 ({lda.explained_variance_ratio_[1]*100:.1f}%)', fontsize=12)
ax1.set_title('LDA - Projection par type de décideur', fontsize=13, fontweight='bold')
ax1.legend(loc='best', fontsize=9)
ax1.grid(True, alpha=0.3)
ax1.axhline(y=0, color='gray', linestyle='--', linewidth=0.5)
ax1.axvline(x=0, color='gray', linestyle='--', linewidth=0.5)

# Matrice de confusion
from sklearn.metrics import confusion_matrix
ax2 = axes[1]
y_pred = lda.predict(X_valid)
cm = confusion_matrix(y_encoded, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=le.classes_, yticklabels=le.classes_, ax=ax2)
ax2.set_xlabel('Prédit')
ax2.set_ylabel('Réel')
ax2.set_title('Matrice de confusion', fontsize=13, fontweight='bold')
plt.setp(ax2.get_xticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.show()

In [ ]:
%%sql -r df_reponses
-- Chargement réponses brutes pour optimisation
SELECT * FROM SNOWFLAKE_LEARNING_DB.PUBLIC.PERSONAS_REPONSES_QUESTIONNAIRE

In [ ]:
# ============================================================================
# 5. OPTIMISATION DU NOMBRE DE QUESTIONS
# ============================================================================
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Questions (hors Q10 qui est la cible)
question_cols = ['Q1_FRIGO', 'Q2_REUNION', 'Q3_SERIE', 'Q4_METEO', 'Q5_RAPPORT', 
                 'Q6_GPS', 'Q7_STAGIAIRE', 'Q8_JEU', 'Q9_FOURNISSEUR',
                 'Q11_PHRASE', 'Q12_PROCESS', 'Q13_FILM', 'Q14_ALARME', 'Q15_DASHBOARD']

# Préparation
df_q = df_reponses[question_cols + ['Q10_DECISION']].copy()
df_q = df_q.dropna()

le_target = LabelEncoder()
y_q = le_target.fit_transform(df_q['Q10_DECISION'])

# One-hot encoding des réponses
X_encoded = pd.get_dummies(df_q[question_cols], prefix=question_cols)

# Score de base avec toutes les questions
lda_q = LinearDiscriminantAnalysis()
base_score = cross_val_score(lda_q, X_encoded, y_q, cv=5).mean()

print("OPTIMISATION DU NOMBRE DE QUESTIONS")
print("="*50)
print(f"\nScore avec 14 questions: {base_score*100:.1f}%")

# Impact de chaque question (leave-one-out)
print(f"\nImpact de chaque question (si retirée):")
print(f"{'Question':>15} {'Impact':>10}")
print("-"*27)

results = []
for q in question_cols:
    cols_without_q = [c for c in question_cols if c != q]
    X_temp = pd.get_dummies(df_q[cols_without_q], prefix=cols_without_q)
    score = cross_val_score(LinearDiscriminantAnalysis(), X_temp, y_q, cv=5).mean()
    impact = base_score - score
    results.append({'question': q, 'score_sans': score, 'impact': impact})

df_importance = pd.DataFrame(results).sort_values('impact', ascending=False)

for _, row in df_importance.iterrows():
    impact_pct = row['impact'] * 100
    bar = '█' * int(abs(impact_pct) * 5) if impact_pct > 0 else ''
    print(f"{row['question']:>15} {impact_pct:>+8.1f}% {bar}")

print(f"\n→ Top 5 questions les plus prédictives:")
for q in df_importance.head(5)['question']:
    print(f"   - {q}")

In [ ]:
# Courbe score vs nombre de questions
top_questions = df_importance.head(10)['question'].tolist()

print("SCORE VS NOMBRE DE QUESTIONS")
print("="*50)

scores_by_n = []
for n in range(1, len(top_questions) + 1):
    selected = top_questions[:n]
    X_n = pd.get_dummies(df_q[selected], prefix=selected)
    score_n = cross_val_score(LinearDiscriminantAnalysis(), X_n, y_q, cv=5).mean()
    scores_by_n.append({'n': n, 'score': score_n, 'questions': selected})
    print(f"{n:2} question(s): {score_n*100:5.1f}%")

# Graphique
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(range(1, len(scores_by_n)+1), [s['score']*100 for s in scores_by_n], 
        'bo-', linewidth=2, markersize=10)
ax.axhline(y=base_score*100, color='red', linestyle='--', 
           label=f'14 questions ({base_score*100:.1f}%)')
ax.axhline(y=70, color='green', linestyle=':', alpha=0.7, label='Seuil 70%')
ax.set_xlabel('Nombre de questions', fontsize=12)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Performance vs Nombre de questions', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xticks(range(1, len(scores_by_n)+1))
plt.tight_layout()
plt.show()

# Recommandation
optimal = next((s for s in scores_by_n if s['score'] >= 0.65), scores_by_n[-1])
print(f"\n→ RECOMMANDATION: {optimal['n']} questions pour {optimal['score']*100:.1f}% d'accuracy")
print(f"   Questions: {', '.join([q.split('_')[0] for q in optimal['questions']])}")

In [ ]:
# ============================================================================
# 6. SOUS-SEGMENTATION PAR TYPE DE DÉCIDEUR
# ============================================================================
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

print("SOUS-SEGMENTATION PAR TYPE DE DÉCIDEUR")
print("="*60)

df_full = df.copy()
df_full = df_full[df_full['TYPE_DECIDEUR'].notna()]

for decideur_type in df_full['TYPE_DECIDEUR'].unique():
    mask = df_full['TYPE_DECIDEUR'] == decideur_type
    subset = df_full[mask]
    n = len(subset)
    
    if n < 15:
        continue
    
    print(f"\n{'='*60}")
    print(f"{decideur_type.upper()} (n={n})")
    print(f"{'='*60}")
    
    X_sub = subset[['AXE_A', 'AXE_B', 'AXE_C', 'AXE_D']].values
    
    # Recherche du meilleur k
    best_k = 2
    best_sil = -1
    for k in range(2, min(5, n//5)):
        km = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = km.fit_predict(X_sub)
        if len(set(labels)) > 1:
            sil = silhouette_score(X_sub, labels)
            if sil > best_sil:
                best_sil = sil
                best_k = k
    
    if best_sil > 0.2:
        km = KMeans(n_clusters=best_k, random_state=42, n_init=10)
        labels = km.fit_predict(X_sub)
        
        print(f"\n→ {best_k} SOUS-GROUPES (silhouette={best_sil:.2f})")
        
        for c in range(best_k):
            c_mask = labels == c
            c_data = subset.iloc[c_mask]
            print(f"\n   Sous-groupe {c+1} (n={c_mask.sum()}):")
            print(f"      A={c_data['AXE_A'].mean():.1f} B={c_data['AXE_B'].mean():.1f} C={c_data['AXE_C'].mean():.1f} D={c_data['AXE_D'].mean():.1f}")
            top_arch = c_data['ARCHETYPE'].value_counts().head(2)
            archs = ', '.join([f"{a.replace('Le ', '')}({v})" for a, v in top_arch.items()])
            print(f"      Archétypes: {archs}")
    else:
        print(f"\n→ GROUPE HOMOGÈNE")

## Résumé

### Résultats clés

| Analyse | Résultat |
|---------|----------|
| ACP - Variance PC1+PC2 | ~95% |
| K-Means - Clusters identifiés | 4 |
| LDA - Accuracy | ~72% |
| Questions minimum | 3 (70% accuracy) |
| Sous-segments totaux | 8 |

### Prochaines étapes

1. Déployer le questionnaire (3 questions) sur formulaire de capture
2. Scorer automatiquement chaque nouveau lead
3. Router vers la séquence email appropriée
4. Mesurer l'impact sur le taux de conversion